## 1. Анализ бронирования зала

**Источник данных:** два файла Excel с ручным ведением броней  
**Проблемы при загрузке:** смешанные форматы дат (строки, числа, datetime),  
служебные пометки в колонке дат, отсутствующий год в части записей  
**Итоговый датасет:** 1414 броней, C января по май, 2026 г.

In [ ]:
import pandas as pd

# Загружаем сырые данные
df_raw = pd.read_excel('../data/Брони искусство 3.0.xlsx',
                       sheet_name='Брони 3.0',
                       header=None)

print('Размер таблицы:', df_raw.shape)
print()
print('Первые 20 строк:')
df_raw.head(20)

In [ ]:
bookings = []
current_date = None

for _, row in df_raw.iterrows():
    
    # Если в колонке 0 есть значение — это строка с датой
    if pd.notna(row[0]):
        current_date = row[0]
        continue
    
    # Если имя (колонка 2) есть и это не заголовок — это бронь
    name = row[2]
    if pd.isna(name) or name in ['Воскресенье ', 'Понедельник', 'Вторник',
                                  'Среда', 'Четверг', 'Пятница', 'Суббота',
                                  'ЧЕТВЕРГ', 'ПЯТНИЦА', 'СУББОТА', 'ВОСКРЕСЕНЬЕ',
                                  'ПОНЕДЕЛЬНИК', 'ВТОРНИК', 'СРЕДА']:
        continue
    
    bookings.append({
        'date':   current_date,
        'time':   row[1],
        'name':   name,
        'table':  row[3],
        'guests': row[4],
        'phone':  row[5],
        'notes':  row[6],
    })

df = pd.DataFrame(bookings)

print('Строк после парсинга:', len(df))
print()
df.head(10)

In [ ]:
# Конвертируем дату в datetime
df['date'] = pd.to_datetime(df['date'], dayfirst = True, errors = 'coerce')

# Конвертируем количество гостей в число
# errors = 'coerce' — всё что не число превратит в NaN
df['guests'] = pd.to_numeric(df['guests'], errors = 'coerce')

# Добавляем день недели и номер часа
df['weekday'] = df['date'].dt.day_name()
df['month']   = df['date'].dt.month

print('Типы колонок:')
print(df.dtypes)
print()
print('Пропуски по колонкам:')
print(df.isnull().sum())
print()
print(f'Период данных: {df["date"].min().date()} — {df["date"].max().date()}')
print(f'Всего броней: {len(df)}')

In [ ]:
# Все уникальные значения из колонки 0 сырых данных, это строки-заголовки с датами
dates_raw = df_raw[0].dropna().unique()
print(f'Всего уникальных значений в колонке дат: {len(dates_raw)}')
print()
print('Первые 20 строк:')
for d in dates_raw[:20]:
    print(repr(d), type(d))

In [ ]:
def parse_date(val):
    if pd.isna(val):
        return pd.NaT
    
    # Если строка — парсим напрямую
    if isinstance(val, str):
        try:
            return pd.to_datetime(val, dayfirst = True)
        except:
            return pd.NaT
    
    # Если число, например 11.01, — превращаем в строку и добавляем год
    if isinstance(val, float):
        # 11.01 → '11.01' → '11.01.2026'
        s = str(val)  # '11.01'
        try:
            return pd.to_datetime(s + '.2026', dayfirst = True)
        except:
            return pd.NaT
    
    return pd.NaT

# Применяем к сырым данным заново
# Пересобираем df с исправленными датами
bookings2 = []
current_date = None

for _, row in df_raw.iterrows():
    if pd.notna(row[0]):
        current_date = parse_date(row[0])
        continue
    
    name = row[2]
    if pd.isna(name) or str(name).strip().upper() in [
        'ЧЕТВЕРГ','ПЯТНИЦА','СУББОТА','ВОСКРЕСЕНЬЕ',
        'ПОНЕДЕЛЬНИК','ВТОРНИК','СРЕДА',
        'СТОЛ', 'ТАЙМИНГ 2,5 ЧАСА'
    ]:
        continue
    
    bookings2.append({
        'date':   current_date,
        'time':   row[1],
        'name':   str(name).strip(),
        'table':  row[3],
        'guests': row[4],
        'phone':  row[5],
        'notes':  row[6],
    })

df = pd.DataFrame(bookings2)
df['guests']  = pd.to_numeric(df['guests'], errors = 'coerce')
df['weekday'] = df['date'].dt.day_name()
df['month']   = df['date'].dt.month

print('Пропуски в дате:', df['date'].isna().sum())
print(f'Период: {df["date"].min().date()} — {df["date"].max().date()}')
print(f'Всего броней: {len(df)}')

In [ ]:
# Смотрим строки где дата не распозналась
missing_dates = df[df['date'].isna()]
print(f'Строк без даты: {len(missing_dates)}')
print()
print('Примеры имён из этих строк:')
print(missing_dates['name'].value_counts().head(20))

In [ ]:
# Смотрим все уникальные значения в колонке 0
# которые не распознались как дата
problem_vals = []

for val in df_raw[0].dropna().unique():
    result = parse_date(val)
    if pd.isna(result):
        problem_vals.append((repr(val), type(val).__name__))

print(f'Нераспознанных значений: {len(problem_vals)}')
print()
for v, t in problem_vals:
    print(f'  {t:10} | {v}')

In [ ]:
# Удаляем строки без распознанной даты
df = df.dropna(subset = ['date'])

print(f'Броней после очистки: {len(df)}')
print(f'Период: {df["date"].min().date()} — {df["date"].max().date()}')
print()
print('Распределение по месяцам:')
print(df['month'].value_counts().sort_index())
print()
print('Пропуски по колонкам:')
print(df.isnull().sum())

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker

# Задаём порядок дней
day_order = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']
day_labels = ['Пн', 'Вт', 'Ср', 'Чт', 'Пт', 'Сб', 'Вс']

# Считаем брони по дням
by_day = df['weekday'].value_counts().reindex(day_order).fillna(0)

fig, ax = plt.subplots(figsize=(10, 5))
bars = ax.bar(day_labels, by_day.values, color='steelblue', edgecolor='white', linewidth=0.5)

# Подписи на барах
for bar, val in zip(bars, by_day.values):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 3,
            str(int(val)), ha = 'center', va = 'bottom', fontsize = 11)

ax.set_title('Количество броней по дням недели', fontsize = 14, pad = 15)
ax.set_xlabel('ДЕНЬ НЕДЕЛИ')
ax.set_ylabel('КОЛИЧЕСТВО БРОНЕЙ')
ax.set_ylim(0, by_day.max() * 1.15)
ax.yaxis.set_major_locator(ticker.MultipleLocator(20))
ax.grid(axis = 'y', alpha = 0.3)
ax.spines[['top', 'right']].set_visible(False)

plt.tight_layout()
plt.show()

print('\nЦифры:')
for day, label, val in zip(day_order, day_labels, by_day.values):
    print(f'  {label}: {int(val)}')

In [ ]:
# Смотрим форматы в колонке time
print('Примеры значений в time:')
print(df['time'].dropna().unique()[:30])

In [ ]:
import re

def parse_time(val):
    if pd.isna(val):
        return None
    
    # Уже объект времени — берём час напрямую
    if hasattr(val, 'hour'):
        return val.hour
    
    # Строка — чистим и берём первое время
    s = str(val).strip()
    
    # Если диапазон типа '20-30/21-00' — берём первую часть
    s = s.split('/')[0].strip()
    
    # Ищем часы: '20-00', '20:00', '20.00'
    match = re.match(r'(\d{1,2})[-:.](\d{2})', s)
    if match:
        return int(match.group(1))
    
    return None

df['hour'] = df['time'].apply(parse_time)

print('Пропуски в hour:', df['hour'].isna().sum())
print()
print('Распределение по часам:')
print(df['hour'].value_counts().sort_index())

In [ ]:
# Убираем ночные часы 0-3 — это продолжение предыдущего вечера, не новый рабочий день
# Оставляем реальное время прихода: с 12 до 23
df_evening = df[df['hour'] >= 12].copy()

hours = sorted(df_evening['hour'].dropna().unique())
counts = df_evening['hour'].value_counts().reindex(hours).fillna(0)

fig, ax = plt.subplots(figsize=(12, 5))
bars = ax.bar([str(int(h)) + ':00' for h in hours],
              counts.values,
              color='steelblue', edgecolor='white', linewidth=0.5)

for bar, val in zip(bars, counts.values):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1.5,
            str(int(val)), ha='center', va='bottom', fontsize=10)

ax.set_title('Распределение броней по времени прихода', fontsize=14, pad=15)
ax.set_xlabel('Время')
ax.set_ylabel('Количество броней')
ax.set_ylim(0, counts.max() * 1.15)
ax.grid(axis='y', alpha=0.3)
ax.spines[['top', 'right']].set_visible(False)

plt.tight_layout()
plt.show()

In [ ]:
import numpy as np

# Строим сводную таблицу: строки = дни недели, колонки = часы
day_order  = ['Monday','Tuesday','Wednesday','Thursday','Friday','Saturday','Sunday']
day_labels = ['Пн','Вт','Ср','Чт','Пт','Сб','Вс']

pivot = df_evening.groupby(['weekday','hour']).size().unstack(fill_value=0)
pivot = pivot.reindex(day_order).fillna(0)

fig, ax = plt.subplots(figsize=(14, 5))
im = ax.imshow(pivot.values, aspect='auto', cmap='YlOrRd')

# Оси
ax.set_xticks(range(len(pivot.columns)))
ax.set_xticklabels([f'{int(h)}:00' for h in pivot.columns])
ax.set_yticks(range(len(day_labels)))
ax.set_yticklabels(day_labels)

# Значения в ячейках
for i in range(len(pivot.index)):
    for j in range(len(pivot.columns)):
        val = int(pivot.values[i, j])
        color = 'white' if val > pivot.values.max() * 0.6 else 'black'
        ax.text(j, i, str(val), ha='center', va='center',
                fontsize=9, color=color)

plt.colorbar(im, ax=ax, label='Количество броней')
ax.set_title('Загрузка по дням недели и времени', fontsize=14, pad=15)
plt.tight_layout()
plt.show()

In [ ]:
# Ищем все пометки о неявках
print('Все уникальные значения в notes:')
print(df['notes'].dropna().value_counts())

In [ ]:
# Ключевые слова для неявок
no_show_keywords = ['не пришл', 'не пришел', 'не пришёл', 'не пришла', 'не пришли',
                    'no show', 'отмен', 'не явил', 'трубку не берёт', 'не берет трубки', 'не берет',
                    'не отвечает', 'ушли', 'передумали', 'в другом', 'в другое', 'не придут', 'переносит', 
                    'перенеcли', 'заболел', 'планы', 'трубку', 'сбросила трубку']

# Создаём флаг no_show
df['no_show'] = df['notes'].str.lower().str.contains(
    '|'.join(no_show_keywords), na=False
)

# Смотрим что нашли
print(f'Всего неявок: {df["no_show"].sum()}')
print(f'Всего броней: {len(df)}')
print(f'Процент неявок: {df["no_show"].mean()*100:.1f}%')
print()
print('Найденные записи о неявках:')
print(df[df['no_show']]['notes'].value_counts())

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle('Анализ броней — Ресторан Искусство, январь – март, 2026',
             fontsize=14, fontweight='bold', y=1.02)

# ── График 1: по дням недели ──────────────────────────────
by_day = df['weekday'].value_counts().reindex(day_order).fillna(0)
bars1 = axes[0].bar(day_labels, by_day.values,
                    color='steelblue', edgecolor='white')
for bar, val in zip(bars1, by_day.values):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 2,
                 str(int(val)), ha='center', fontsize=9)
axes[0].set_title('Брони по дням недели')
axes[0].set_ylabel('Количество броней')
axes[0].grid(axis='y', alpha=0.3)
axes[0].spines[['top','right']].set_visible(False)

# ── График 2: по часам ────────────────────────────────────
counts_h = df_evening['hour'].value_counts().sort_index()
bars2 = axes[1].bar([f'{int(h)}:00' for h in counts_h.index],
                    counts_h.values,
                    color='steelblue', edgecolor='white')
for bar, val in zip(bars2, counts_h.values):
    axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1,
                 str(int(val)), ha='center', fontsize=8)
axes[1].set_title('Брони по времени прихода')
axes[1].set_ylabel('Количество броней')
axes[1].tick_params(axis='x', rotation=45)
axes[1].grid(axis='y', alpha=0.3)
axes[1].spines[['top','right']].set_visible(False)

# ── График 3: Неявки по дням недели ─────────────────────
noshow_by_day = df[df['no_show']]['weekday'].value_counts().reindex(day_order).fillna(0)
bars3 = axes[2].bar(day_labels, noshow_by_day.values,
                    color='salmon', edgecolor='white')
for bar, val in zip(bars3, noshow_by_day.values):
    if val > 0:
        axes[2].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.1,
                     str(int(val)), ha='center', fontsize=9)
axes[2].set_title(f'Неявки по дням (всего: {df["no_show"].sum()}, {df["no_show"].mean()*100:.1f}%)')
axes[2].set_ylabel('Количество неявок')
axes[2].grid(axis='y', alpha=0.3)
axes[2].spines[['top','right']].set_visible(False)

plt.tight_layout()
plt.show()

## Анализ второго файла бронирования столов

In [ ]:
df_raw2 = pd.read_excel('../data/Брони 4.0.xlsx',
                        sheet_name='Брони',
                        header=None)

print('Размер:', df_raw2.shape)
print()
print('Первые 20 строк:')
df_raw2.head(20)

In [ ]:
dates_raw2 = df_raw[0].dropna().unique()
print('уникальные значения в колонке дат:')
for d in dates_raw2:
    print(repr(d), type(d))

In [ ]:
# Парсим второй файл 
bookings3 = []
current_date = None

for _, row in df_raw2.iterrows():
    if pd.notna(row[0]):
        current_date = parse_date(row[0])
        continue

    name = row[2]
    if pd.isna(name) or str(name).strip().upper() in [
        'ЧЕТВЕРГ','ПЯТНИЦА','СУББОТА','ВОСКРЕСЕНЬЕ',
        'ПОНЕДЕЛЬНИК','ВТОРНИК','СРЕДА',
        'СТОЛ','ТАЙМИНГ','КОЛ-ВО','ТЕЛЕФОН',
        'ВОСКРЕСЕНЬЕ ','Б'
    ]:
        continue

    bookings3.append({
        'date':   current_date,
        'time':   row[1],
        'name':   str(name).strip(),
        'table':  row[3],
        'guests': row[4],
        'phone':  row[5],
        'notes':  row[6],
    })

df2 = pd.DataFrame(bookings3)
df2['guests']  = pd.to_numeric(df2['guests'], errors='coerce')
df2 = df2.dropna(subset=['date'])
df2['weekday'] = df2['date'].dt.day_name()
df2['month']   = df2['date'].dt.month
df2['hour']    = df2['time'].apply(parse_time)
df2['no_show'] = df2['notes'].str.lower().str.contains(
    '|'.join(no_show_keywords), na=False
)

print(f'Броней в файле 2: {len(df2)}')
print(f'Период: {df2["date"].min().date()} — {df2["date"].max().date()}')
print()

# Объединяем оба файла
df_all = pd.concat([df, df2], ignore_index=True)

# Убираем дубликаты — одинаковые дата+имя+время
df_all = df_all.drop_duplicates(subset=['date','name','time'])

print(f'Итого броней после объединения: {len(df_all)}')
print(f'Период: {df_all["date"].min().date()} — {df_all["date"].max().date()}')
print()
print('По месяцам:')
print(df_all['month'].value_counts().sort_index())

In [ ]:
# Смотрим первые 25 строк сырого файла
print(df_raw2.head(25).to_string())
print()
# И проверяем что вообще попадает в current_date
count = 0
for _, row in df_raw2.iterrows():
    if pd.notna(row[0]):
        d = parse_date(row[0])
        print(f'row[0]={repr(row[0])} → parse_date={d}')
        count += 1
        if count > 5:
            break

In [ ]:
def parse_date(val):
    if pd.isna(val):
        return pd.NaT
    
    # Уже объект datetime — возвращаем напрямую
    if isinstance(val, (pd.Timestamp, __import__('datetime').datetime)):
        return pd.Timestamp(val)
    
    # Строка
    if isinstance(val, str):
        try:
            return pd.to_datetime(val, dayfirst=True)
        except:
            return pd.NaT
    
    # Число (например 11.01)
    if isinstance(val, float):
        s = str(val)
        try:
            return pd.to_datetime(s + '.2026', dayfirst=True)
        except:
            return pd.NaT
    
    return pd.NaT

# Проверяем
import datetime
test_vals = [
    datetime.datetime(2026, 3, 30, 0, 0),
    '1.01.2026',
    11.01,
    '*'
]
for v in test_vals:
    print(f'{repr(v):45} → {parse_date(v)}')

In [ ]:
# Парсим второй файл заново
bookings3 = []
current_date = None

for _, row in df_raw2.iterrows():
    if pd.notna(row[0]):
        current_date = parse_date(row[0])
        continue

    name = row[2]
    if pd.isna(name) or str(name).strip().upper() in [
        'ЧЕТВЕРГ','ПЯТНИЦА','СУББОТА','ВОСКРЕСЕНЬЕ',
        'ПОНЕДЕЛЬНИК','ВТОРНИК','СРЕДА',
        'СТОЛ','ТАЙМИНГ','КОЛ-ВО','ТЕЛЕФОН',
        'ВОСКРЕСЕНЬЕ ','Б'
    ]:
        continue

    bookings3.append({
        'date':   current_date,
        'time':   row[1],
        'name':   str(name).strip(),
        'table':  row[3],
        'guests': row[4],
        'phone':  row[5],
        'notes':  row[6],
    })

df2 = pd.DataFrame(bookings3)
df2['guests']  = pd.to_numeric(df2['guests'], errors='coerce')
df2 = df2.dropna(subset=['date'])
df2['weekday'] = df2['date'].dt.day_name()
df2['month']   = df2['date'].dt.month
df2['hour']    = df2['time'].apply(parse_time)
df2['no_show'] = df2['notes'].str.lower().str.contains(
    '|'.join(no_show_keywords), na=False
)

print(f'Броней в файле 2: {len(df2)}')
print(f'Период: {df2["date"].min().date()} — {df2["date"].max().date()}')
print()

# Объединяем
df_all = pd.concat([df, df2], ignore_index=True)
df_all = df_all.drop_duplicates(subset=['date', 'name', 'time'])

print(f'Итого броней: {len(df_all)}')
print(f'Период: {df_all["date"].min().date()} — {df_all["date"].max().date()}')
print()
print('По месяцам:')
print(df_all['month'].value_counts().sort_index())

In [ ]:
print('Файл 1 (Брони 3.0):')
print(df['date'].dt.month.value_counts().sort_index())
print()
print('Файл 2 (Брони 4.0):')
print(df2['date'].dt.month.value_counts().sort_index())
print()

# Сколько записей пересекается
overlap = pd.merge(df, df2, on=['date','name'], suffixes=('_1','_2'))
print(f'Пересечений по дате+имени: {len(overlap)}')

In [ ]:
bookings_1 = []
current_date = None

for _, row in df_raw.iterrows():
    if pd.notna(row[0]):
        current_date = parse_date(row[0])
        continue

    name = row[2]
    if pd.isna(name) or str(name).strip().upper() in [
        'ЧЕТВЕРГ','ПЯТНИЦА','СУББОТА','ВОСКРЕСЕНЬЕ',
        'ПОНЕДЕЛЬНИК','ВТОРНИК','СРЕДА',
        'СТОЛ','ТАЙМИНГ','КОЛ-ВО','ТЕЛЕФОН',
        'ВОСКРЕСЕНЬЕ ','Б'
    ]:
        continue

    bookings_1.append({
        'date':   current_date,
        'time':   row[1],
        'name':   str(name).strip(),
        'table':  row[3],
        'guests': row[4],
        'phone':  row[5],
        'notes':  row[6],
    })

df1 = pd.DataFrame(bookings_1)
df1['guests']  = pd.to_numeric(df1['guests'], errors='coerce')
df1 = df1.dropna(subset=['date'])
df1['weekday'] = df1['date'].dt.day_name()
df1['month']   = df1['date'].dt.month
df1['hour']    = df1['time'].apply(parse_time)
df1['no_show'] = df1['notes'].str.lower().str.contains(
    '|'.join(no_show_keywords), na=False
)

print(f'Файл 1: {len(df1)} броней')
print(f'Период: {df1["date"].min().date()} — {df1["date"].max().date()}')
print('По месяцам:')
print(df1['date'].dt.month.value_counts().sort_index())

In [ ]:
df_all = pd.concat([df1, df2], ignore_index=True)

import hashlib

def anonymize_name(name):
    h = hashlib.md5(name.encode()).hexdigest()[:6].upper()
    return f'Гость_{h}'

df_all['name']  = df_all['name'].apply(anonymize_name)
df_all['phone'] = '***'
print(f'Итого броней: {len(df_all)}')
print(f'Период: {df_all["date"].min().date()} — {df_all["date"].max().date()}')
print()
print('По месяцам:')
print(df_all['date'].dt.month.value_counts().sort_index())

In [ ]:
df_evening_all = df_all[df_all['hour'] >= 12].copy()

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle('Анализ броней — Ресторан Искусство, январь – май, 2026',
             fontsize=14, fontweight='bold', y=1.02)

# ── График 1: по дням недели ──────────────────────────────
by_day = df_all['weekday'].value_counts().reindex(day_order).fillna(0)
bars1 = axes[0].bar(day_labels, by_day.values,
                    color='steelblue', edgecolor='white')
for bar, val in zip(bars1, by_day.values):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 2,
                 str(int(val)), ha='center', fontsize=9)
axes[0].set_title('Брони по дням недели')
axes[0].set_ylabel('Количество броней')
axes[0].grid(axis='y', alpha=0.3)
axes[0].spines[['top','right']].set_visible(False)

# ── График 2: по часам ────────────────────────────────────
counts_h = df_evening_all['hour'].value_counts().sort_index()
bars2 = axes[1].bar([f'{int(h)}:00' for h in counts_h.index],
                    counts_h.values,
                    color='steelblue', edgecolor='white')
for bar, val in zip(bars2, counts_h.values):
    axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1,
                 str(int(val)), ha='center', fontsize=8)
axes[1].set_title('Брони по времени прихода')
axes[1].set_ylabel('Количество броней')
axes[1].tick_params(axis='x', rotation=45)
axes[1].grid(axis='y', alpha=0.3)
axes[1].spines[['top','right']].set_visible(False)

# ── График 3: no-show ─────────────────────────────────────
noshow_by_day = df_all[df_all['no_show']]['weekday'].value_counts().reindex(day_order).fillna(0)
bars3 = axes[2].bar(day_labels, noshow_by_day.values,
                    color='salmon', edgecolor='white')
for bar, val in zip(bars3, noshow_by_day.values):
    if val > 0:
        axes[2].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.1,
                     str(int(val)), ha='center', fontsize=9)
axes[2].set_title(f'Неявки по дням (всего: {df_all["no_show"].sum()}, {df_all["no_show"].mean()*100:.1f}%)')
axes[2].set_ylabel('Количество неявок')
axes[2].grid(axis='y', alpha=0.3)
axes[2].spines[['top','right']].set_visible(False)

plt.tight_layout()
plt.show()

# Итоговые цифры
print(f'Всего броней: {len(df_all)}')
print(f'Средний размер компании: {df_all["guests"].mean():.1f} чел.')
print(f'Неявок: {df_all["no_show"].sum()} ({df_all["no_show"].mean()*100:.1f}%)')

In [ ]:
pivot = df_all[df_all['hour'] >= 12].groupby(
    ['weekday','hour']
).size().unstack(fill_value=0)
pivot = pivot.reindex(day_order).fillna(0)

fig, ax = plt.subplots(figsize=(16, 5))
im = ax.imshow(pivot.values, aspect='auto', cmap='YlOrRd')

ax.set_xticks(range(len(pivot.columns)))
ax.set_xticklabels([f'{int(h)}:00' for h in pivot.columns], fontsize=10)
ax.set_yticks(range(len(day_labels)))
ax.set_yticklabels(day_labels, fontsize=11)

for i in range(len(pivot.index)):
    for j in range(len(pivot.columns)):
        val = int(pivot.values[i, j])
        color = 'white' if val > pivot.values.max() * 0.6 else 'black'
        ax.text(j, i, str(val), ha='center', va='center',
                fontsize=10, color=color, fontweight='bold')

plt.colorbar(im, ax=ax, label='Количество броней')
ax.set_title('Загрузка по дням недели и времени  |  январь – май, 2026',
             fontsize=14, pad=15)
plt.tight_layout()
plt.show()

## Выводы  по анализу бронирования зала

**Данные:** 1414 бронях, январь–май 2026, ресторан «Искусство», Уфа

### Загрузка по дням
- Пятница — пик броней - 356, в 3.5 раза больше вторника (104)
- Выходные (Сб 290, Вс 224) уступают пятнице
- Вторник — самый тихий день (104 броней)
- **Вывод:** основная выручка делается в пятницу, не в субботу

### Загрузка по времени
- Пиковая посадка: 20:00 (259) и 21:00 (251)
- С 18:00 возрастает количество посетителей
- До 17:00 посадка минимальная или почти отсутствует
- **Вывод:** Требуется усиление персонала с 19:00 до 22:00

### Самые горячие слоты по heatmap
- Пт 20:00 — 68 броней, Пт 19:00 — 58, Сб 20:00 — 60
- Воскресенье активнее с 16:00 — гости приходят раньше
- **Вывод:** в пятницу с 19 до 21 нужен полный состав

### Неявки
- 76 неявки из 1414 броней = 5.4 % — хороший показатель
- Больше всего неявок в пятницу (26) — самый загруженный день
- Самый неприбыльный день - вторник (104 броней)
- **Вывод:** Предложение оптимизации работы персоналы:
- в дни простоя уменьшения количества рабочих часов и количества человек в часы наименьшего количества посетителей;
- в дни загрузки обеспечить выход большего числа сотрудников, на часы пиковой посадки посетителей;
- правильная организация алгоритма работы официантов и поваров для снижения недопонимания и быстрого и качественного обслуживания гостей.